In [9]:
from langchain_community.graphs import Neo4jGraph
from langchain_openai import ChatOpenAI
from langchain_classic.chains import GraphCypherQAChain
from pinecone import Pinecone
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

load_dotenv()


True

In [5]:
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')

graph = Neo4jGraph(
    url="neo4j+s://f5886a71.databases.neo4j.io",
    username="neo4j",
    password="fFCqS5-Ak3ZLppY0p1e2rrL1ZN_FJwt7oH_McxlCh5o"
)

pc = Pinecone(api_key=PINECONE_API_KEY)
INDEX_NAME = "lol-rag"  
pinecone_index = pc.Index(INDEX_NAME)


openai_client = OpenAI(api_key=OPENAI_API_KEY)

llm = ChatOpenAI(
    model_name="gpt-4",
    temperature=0, 
    api_key=OPENAI_API_KEY
)

graph.refresh_schema()


In [6]:
def create_embedding(text):
    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding


In [7]:
def pinecone_search(query, top_k=10, filter_type=None):
    """
    Semantic search using Pinecone.
    
    Args:
        query: Natural language query
        top_k: Number of results
        filter_type: 'ability' or 'champion' or None
    
    Returns:
        List of matches with metadata
    """
    query_embedding = create_embedding(query)
    
    query_filter = None
    if filter_type:
        query_filter = {'type': filter_type}
    
    results = pinecone_index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True,
        filter=query_filter
    )
    
    return results['matches']

print("✓ Pinecone search function ready")

✓ Pinecone search function ready


In [8]:
# Create GraphCypherQAChain for Neo4j queries
neo4j_chain = GraphCypherQAChain.from_llm(
    graph=graph,
    llm=llm,
    verbose=True,
    allow_dangerous_requests=True,
    return_intermediate_steps=True  # Important: gives us the Cypher query
)

print("✓ Neo4j chain ready")

✓ Neo4j chain ready


In [10]:
def analyze_query(question):
    """
    Analyze user question and determine retrieval strategy.
    
    Returns:
        dict with:
        - strategy: 'neo4j', 'pinecone', or 'hybrid'
        - reasoning: Why this strategy
        - neo4j_hints: Suggested Cypher patterns (if applicable)
        - semantic_query: Reformulated query for Pinecone (if applicable)
    """
    
    analysis_prompt = f"""
You are a query analyzer for a League of Legends knowledge base with two databases:

**Neo4j (Graph Database):**
- Structured data: roles, damage types, crowd control, scaling stats
- Good for: exact filters, boolean logic, relationships
- Schema: {graph.schema[:800]}

**Pinecone (Vector Database):**
- Semantic search on ability descriptions and champion profiles
- Good for: conceptual queries, themes, similarity
- Examples: "fire abilities", "tanky champions", "burst damage"

Analyze this question: "{question}"

Respond in JSON format:
{{
    "strategy": "neo4j" | "pinecone" | "hybrid",
    "reasoning": "why you chose this strategy",
    "neo4j_hints": "suggested concepts to query" (if using neo4j),
    "semantic_query": "reformulated query for vector search" (if using pinecone),
    "filters": {{
        "role": "Mage" (example),
        "attack_type": "Ranged" (example)
    }} (if using hybrid)
}}

**Decision Rules:**
- Use Neo4j for: specific roles, attack types, exact damage/CC types, difficulty
- Use Pinecone for: themes (fire/ice), playstyles (burst/poke), vague concepts
- Use Hybrid for: combinations ("ranged mages with fire")
"""
    
    response = openai_client.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": "You are a query strategy analyzer. Always respond with valid JSON."},
            {"role": "user", "content": analysis_prompt}
        ],
        temperature=0
    )
    
    # Parse JSON response
    try:
        analysis = json.loads(response.choices[0].message.content)
        return analysis
    except json.JSONDecodeError:
        # Fallback if JSON parsing fails
        return {
            "strategy": "neo4j",
            "reasoning": "Default fallback",
            "neo4j_hints": question
        }

print("✓ Query analyzer ready")

✓ Query analyzer ready


In [11]:
def retrieve_context(question):
    """
    Master retrieval function.
    
    1. Analyze query
    2. Retrieve from appropriate source(s)
    3. Format context for GPT-4
    
    Returns:
        dict with:
        - context: Formatted text for GPT-4
        - sources: List of sources used
        - strategy: Which strategy was used
    """
    
    # Step 1: Analyze query
    analysis = analyze_query(question)
    strategy = analysis['strategy']
    
    print(f"\n🔍 Strategy: {strategy}")
    print(f"   Reasoning: {analysis['reasoning']}")
    
    context_parts = []
    sources = []
    
    # Step 2: Execute retrieval based on strategy
    
    if strategy == 'neo4j':
        # Pure Neo4j query
        print("\n📊 Querying Neo4j...")
        result = neo4j_chain.invoke({"query": question})
        
        context_parts.append(f"**Neo4j Results:**\n{result['result']}")
        sources.append(f"Neo4j Query: {result.get('intermediate_steps', [{}])[0].get('query', 'N/A')}")
    
    elif strategy == 'pinecone':
        # Pure semantic search
        semantic_query = analysis.get('semantic_query', question)
        print(f"\n🔍 Searching Pinecone: '{semantic_query}'")
        
        matches = pinecone_search(semantic_query, top_k=10)
        
        # Format results
        context_parts.append("**Semantic Search Results:**")
        for i, match in enumerate(matches[:5], 1):
            meta = match['metadata']
            if meta['type'] == 'ability':
                context_parts.append(
                    f"{i}. {meta['champion_name']} - {meta['ability_name']} [{meta['slot']}]\n"
                    f"   Preview: {meta['text'][:150]}..."
                )
            else:
                context_parts.append(
                    f"{i}. {meta['champion_name']} - {meta['title']}\n"
                    f"   Roles: {meta.get('roles', 'N/A')}"
                )
            sources.append(f"Pinecone: {meta['champion_name']}")
    
    elif strategy == 'hybrid':
        # Combined approach
        print("\n🔀 Hybrid Retrieval (Neo4j + Pinecone)")
        
        # First, semantic search
        semantic_query = analysis.get('semantic_query', question)
        print(f"   1. Pinecone: '{semantic_query}'")
        matches = pinecone_search(semantic_query, top_k=15)
        
        # Extract champion IDs from semantic results
        champion_ids = list(set([m['metadata']['champion_id'] for m in matches]))
        
        # Second, filter with Neo4j
        filters = analysis.get('filters', {})
        if filters:
            print(f"   2. Neo4j filters: {filters}")
            
            # Build Cypher query
            cypher_parts = ["MATCH (c:Champion) WHERE c.id IN $champion_ids"]
            params = {'champion_ids': champion_ids}
            
            if 'role' in filters:
                cypher_parts.append("MATCH (c)-[:HAS_ROLE]->(r:Role {name: $role})")
                params['role'] = filters['role']
            
            if 'attack_type' in filters:
                cypher_parts.append("MATCH (c)-[:ATTACKS_WITH]->(at:AttackType {name: $attack_type})")
                params['attack_type'] = filters['attack_type']
            
            cypher_parts.append("RETURN DISTINCT c.id as id, c.name as name")
            cypher_query = " ".join(cypher_parts)
            
            filtered_champions = graph.query(cypher_query, params=params)
            filtered_ids = {c['id'] for c in filtered_champions}
            
            # Filter Pinecone results
            matches = [m for m in matches if m['metadata']['champion_id'] in filtered_ids]
        
        # Format combined results
        context_parts.append("**Hybrid Search Results:**")
        for i, match in enumerate(matches[:5], 1):
            meta = match['metadata']
            if meta['type'] == 'ability':
                context_parts.append(
                    f"{i}. {meta['champion_name']} - {meta['ability_name']} [{meta['slot']}]\n"
                    f"   Similarity: {match['score']:.3f}\n"
                    f"   {meta['text'][:150]}..."
                )
            sources.append(f"Hybrid: {meta['champion_name']}")
    
    # Step 3: Assemble final context
    context = "\n\n".join(context_parts)
    
    return {
        'context': context,
        'sources': sources[:5],  # Limit sources
        'strategy': strategy,
        'analysis': analysis
    }

print("✓ Master retrieval function ready")

✓ Master retrieval function ready


In [12]:
def generate_answer(question, context_data):
    """
    Generate final answer using GPT-4.
    
    Args:
        question: Original user question
        context_data: Retrieved context from retrieve_context()
    
    Returns:
        Natural language answer
    """
    
    system_prompt = """
You are a League of Legends expert assistant.

Your job:
1. Answer questions accurately using the provided context
2. Be concise but informative
3. Cite specific champions/abilities when relevant
4. If context is insufficient, say so honestly
5. Use natural, conversational language

Guidelines:
- Don't make up information not in the context
- If asked about specific numbers (damage, cooldowns), only provide if in context
- Format lists clearly when showing multiple results
- Explain WHY champions/abilities match the query
"""
    
    user_prompt = f"""
Question: {question}

Context Retrieved:
{context_data['context']}

Please provide a clear, accurate answer based on this context.
"""
    
    response = openai_client.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.3  # Slightly creative but mostly factual
    )
    
    return response.choices[0].message.content

print("✓ Answer generator ready")

✓ Answer generator ready


In [13]:
def ask_question(question, verbose=True):
    """
    Main Q&A function - handles everything end-to-end.
    
    Args:
        question: Natural language question
        verbose: Show retrieval details
    
    Returns:
        Answer string
    """
    
    print(f"\n{'='*60}")
    print(f"❓ Question: {question}")
    print(f"{'='*60}")
    
    # Retrieve context
    context_data = retrieve_context(question)
    
    if verbose:
        print(f"\n📦 Context retrieved ({len(context_data['context'])} chars)")
        print(f"   Strategy: {context_data['strategy']}")
        print(f"   Sources: {', '.join(context_data['sources'][:3])}...")
    
    # Generate answer
    print(f"\n🤖 Generating answer...")
    answer = generate_answer(question, context_data)
    
    print(f"\n{'='*60}")
    print(f"💡 ANSWER:")
    print(f"{'='*60}")
    print(answer)
    print(f"{'='*60}\n")
    
    return {
        'question': question,
        'answer': answer,
        'strategy': context_data['strategy'],
        'sources': context_data['sources']
    }

print("✓ Main Q&A function ready")
print("\n🚀 Ready to answer questions!")
print("\nUsage: ask_question('Find ranged mages with crowd control')")

✓ Main Q&A function ready

🚀 Ready to answer questions!

Usage: ask_question('Find ranged mages with crowd control')


In [24]:
ask_question("Find all abilities that stun")


❓ Question: Find all abilities that stun

🔍 Strategy: neo4j
   Reasoning: The query is asking for a specific crowd control type (stun), which is a structured data that can be directly queried from the Neo4j database.

📊 Querying Neo4j...


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (a:Ability)-[:APPLIES]->(cc:CrowdControl)
WHERE cc.name = 'Stun'
RETURN a.name
Full Context:
[{'a.name': 'Trample'}, {'a.name': 'Decimating Smash'}, {'a.name': 'Unstoppable Onslaught'}, {'a.name': 'Pyromania'}, {'a.name': 'Summon: Tibbers'}, {'a.name': "Hero's Entrance"}, {'a.name': 'Pick a Card'}, {'a.name': 'Disdain'}, {'a.name': 'Triumphant Roar'}, {'a.name': 'Buster Shot'}]

> Finished chain.

📦 Context retrieved (103 chars)
   Strategy: neo4j
   Sources: Neo4j Query: MATCH (a:Ability)-[:APPLIES]->(cc:CrowdControl)
WHERE cc.name = 'Stun'
RETURN a.name...

🤖 Generating answer...

💡 ANSWER:
I'm sorry, but I can't provide a list of all abilities that stun in League of Legends based 

{'question': 'Find all abilities that stun',
 'answer': "I'm sorry, but I can't provide a list of all abilities that stun in League of Legends based on the provided context.",
 'strategy': 'neo4j',
 'sources': ["Neo4j Query: MATCH (a:Ability)-[:APPLIES]->(cc:CrowdControl)\nWHERE cc.name = 'Stun'\nRETURN a.name"]}